## Table of contents
- [1. Imports](#imports)
- [2. Data](#data)
- [3. Named Entity Recognition](#ner)
    - [3.1 Specific Data Preparation](#data_prep)
    - [3.2 Model Implementation](#model)
    - [3.3 Model Evaluation](#evaluation)
- [4. Post-processing & Co-occurrence Analysis](#postproc)
- [5. Next steps](#next)

# <a id=imports></a>1. Imports

[Back to TOC](#Table of contents)

In [ ]:
# Basic imports and project helpers
import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
import os, sys
import pandas as pd
import spacy
from spacy import displacy
import joblib
import networkx as nx
import matplotlib.pyplot as plt
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
# add local source folder so we can import helpers
source_code_path = os.path.abspath('../source')
if source_code_path not in sys.path:
    sys.path.append(source_code_path)
from my_utils import *
from visualizations import *
from general_preprocessing import *

# <a id=data></a>2. Data

[Back to TOC](#Table of contents)

In [ ]:
# Load dataset
data_path = os.path.join('..', 'data', 'atlanta_restaurant_slice_2023.csv')
try:
    dataset_original = load_dataset(data_path)
except Exception:
    dataset_original = pd.read_csv(data_path)
dataset = dataset_original.copy()
dataset.head()

# <a id=ner></a>3. Named Entity Recognition

[Back to TOC](#Table of contents)

## <a id=data_prep></a>3.1 Specific Data Preparation

[Back to TOC](#Table of contents)

In [ ]:
# Tokenization and POS tagging using project pipeline
dataset['pos_list'] = dataset['text'].map(lambda content: main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'))
dataset['tokens'] = dataset['text'].map(lambda content: main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True))
# Feature extraction for CRF
dataset['features'] = dataset.apply(lambda row: sent2features(row['tokens'], row['pos_list']), axis=1)
nlp = spacy.load('en_core_web_sm')
equivalence_table = {'PERSON':'-per','NORP':'-grp','FAC':'-fac','ORG':'-org','GPE':'-gpe','LOC':'-geo','DATE':'-date','TIME':'-tim','PRODUCT':'-prod','MONEY':'-mon','QUANTITY':'-qty','CARDINAL':'-card','ORDINAL':'-ord','':' '}
dataset['ner_labels_custom'] = dataset.apply(lambda row: align_bio_to_custom_tokens(row['text'], row['tokens'], nlp, equivalence_table), axis=1)
# Sanity check
dataset[['text','tokens','ner_labels_custom']].head()

## <a id=model></a>3.2 Model Implementation

[Back to TOC](#Table of contents)

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(dataset['features'], dataset['ner_labels_custom'], test_size=0.2, random_state=39)
train_index = list(X_train.index)
test_index = list(X_test.index)
# CRF model
crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
crf.fit(X_train, y_train)
# keep tokens for inspection
X_test_tokens = dataset.loc[test_index, 'tokens']
y_pred = crf.predict(X_test)
# save model
models_dir = os.path.join('..','models')
os.makedirs(models_dir, exist_ok=True)
joblib.dump(crf, os.path.join(models_dir, 'crf_ner_template.pkl'))

## <a id=evaluation></a>3.3 Model Evaluation

[Back to TOC](#Table of contents)

In [ ]:
labels = list(crf.classes_)
if 'O' in labels: labels.remove('O')
print('Weighted F1:', round(metrics.flat_f1_score(y_test, y_pred, average='weighted', labels=labels), 3))
sorted_labels = sorted(labels, key=lambda name: (name[1:], name[0]))
print(sklearn_crfsuite.metrics.flat_classification_report(y_test, y_pred, labels=sorted_labels, digits=3))
# qualitative check
def show_predictions(tokens_series, y_true, y_pred, n=3):
    for i in range(min(n, len(y_pred))):
        tokens = list(tokens_series.iloc[i])
        true = list(y_true[i])
        pred = list(y_pred[i])
        for t, tr, pr in zip(tokens, true, pred):
            print(f'{t}	{tr}	{pr}')
        print('-'*40)
show_predictions(X_test_tokens.reset_index(drop=True), y_test, y_pred, n=3)

# <a id=postproc></a>4. Post-processing & Co-occurrence Analysis

[Back to TOC](#Table of contents)

In [ ]:
def bio_to_spans(tokens, labels):
    spans = []
    current = []
    current_label = None
    for t, lab in zip(tokens, labels):
        if lab.startswith('B-'):
            if current: spans.append((' '.join(current), current_label))
            current = [t]; current_label = lab[2:]
        elif lab.startswith('I-') and current: current.append(t)
        else:
            if current: spans.append((' '.join(current), current_label)); current = []; current_label = None
    if current: spans.append((' '.join(current), current_label))
    return spans
pred_entities = [bio_to_spans(tokens, labels) for tokens, labels in zip(X_test_tokens, y_pred)]
ent_df = pd.DataFrame({'tokens': list(X_test_tokens), 'preds': pred_entities})
from collections import Counter
edges = Counter()
for ents in ent_df['preds']:
    names = [e[0].lower() for e in ents]
    for i in range(len(names)):
        for j in range(i+1, len(names)): edges[(names[i], names[j])] += 1
G = nx.Graph()
for (a,b), w in edges.items():
    if w >= 2: G.add_edge(a, b, weight=w)
print('Graph nodes:', G.number_of_nodes(), 'edges:', G.number_of_edges())
if G.number_of_nodes()>0:
    pos = nx.spring_layout(G, k=0.5)
    plt.figure(figsize=(10,8))
    nx.draw_networkx(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()

# <a id=next></a>5. Next steps
- Tune CRF hyperparameters (c1, c2) with a small grid search
- Expand dish gazetteer and add `in_gazetteer` feature to improve recall
- Consider fine-tuning a transformer-based NER (Hugging Face) if higher performance needed
- Run clustering / community detection on the co-occurrence graph and compare clusters to `categoryName`
- Save final pipeline and provide an inference function for new reviews